# 5. Add Tools

**Goal:** Add Tavily web search and yfinance OHLCV.

In [7]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "agent.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agno.agent import Agent
from agno.models.openai import OpenAIChat
from knowledge import load_settings

RUN_LIVE = True
settings = load_settings()
model = OpenAIChat(
    id=settings["openrouter_model"],
    api_key=settings["openrouter_api_key"],
    base_url="https://openrouter.ai/api/v1",
)
print("Ready. Live model calls:", RUN_LIVE)

Ready. Live model calls: True


In [8]:
async def ask(workshop_agent, question, session_id=None):
    if not RUN_LIVE:
        return "Skipped. Set RUN_LIVE = True for a model call."
    response = await workshop_agent.arun(question, session_id=session_id)
    return response.content

In [ ]:
INSTRUCTIONS = [
    "For company financial questions, search the knowledge base first.",
    "If the knowledge base is not available, then tell the user explicitly that you cannot proceed further. Do extra message or hallunicated answer",
    "For latest figures, use the latest period in the knowledge base and name it.",
    "Do not use web search when the knowledge base answers the question.",
    "Answer PDF questions only from retrieved evidence.",
    "Cite the source filename and physical PDF page.",
    "Say when the evidence is missing.",
]

In [10]:
from knowledge import create_knowledge
knowledge = create_knowledge()
print("Knowledge ready:", knowledge.name)

Knowledge ready: Gravitas Student Knowledge


In [12]:
from tools import create_tools, market_ohlcv

student_tools = create_tools()
print([getattr(item, "name", getattr(item, "__name__", "")) for item in student_tools])

tool_agent = Agent(
    name="Tool Agent",
    model=model,
    knowledge=knowledge,
    search_knowledge=True,
    tools=student_tools,
    instructions=INSTRUCTIONS,
    tool_call_limit=6,
    markdown=True,
    telemetry=False,
)

if RUN_LIVE:
    print(await market_ohlcv("HCLTECH", period="5d", exchange="NSE"))

    print("Agent response: \n", await ask(tool_agent, question="Fetch 5 days HCLTECH stock ohlcv", session_id="123"))



['websearch', 'market_ohlcv', 'calculate_growth']
{'symbol': 'HCLTECH.NS', 'latest_market_day': '2026-09-16', 'rows': [{'date': '2026-09-10', 'open': 1224.8, 'high': 1237.0, 'low': 1196.9, 'close': 1207.0, 'volume': 2197859}, {'date': '2026-09-11', 'open': 1196.0, 'high': 1223.4, 'low': 1195.7, 'close': 1206.1, 'volume': 2272845}, {'date': '2026-09-14', 'open': 1206.1, 'high': 1206.1, 'low': 1206.1, 'close': 1206.1, 'volume': 0}, {'date': '2026-09-15', 'open': 1252.0, 'high': 1290.0, 'low': 1250.0, 'close': 1253.7, 'volume': 4443984}, {'date': '2026-09-16', 'open': 1262.5, 'high': 1284.6, 'low': 1244.9, 'close': 1253.0, 'volume': 3048581}], 'note': 'On a weekend or holiday, latest_market_day is the previous trading day.'}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HCLTECH"}}}
$HCLTECH: No data found, symbol may be delisted


Agent response: 
 I'll fetch the 5-day OHLCV data for HCLTECH stock using the market data tool.Here are the **5-day OHLCV** details for **HCLTECH** (NSE: HCLTECH.NS):

| Date | Open (₹) | High (₹) | Low (₹) | Close (₹) | Volume |
|------------|----------|----------|---------|-----------|-----------|
| 2026-09-10 | 1,224.80 | 1,237.00 | 1,196.90 | 1,207.00 | 2,197,859 |
| 2026-09-11 | 1,196.00 | 1,223.40 | 1,195.70 | 1,206.10 | 2,272,845 |
| 2026-09-14 | 1,206.10 | 1,206.10 | 1,206.10 | 1,206.10 | 0 |
| 2026-09-15 | 1,252.00 | 1,290.00 | 1,250.00 | 1,253.70 | 4,443,984 |
| 2026-09-16 | 1,262.50 | 1,284.60 | 1,244.90 | 1,253.00 | 3,048,581 |

> **Note:** The latest market day is **2026-09-16**. On **2026-09-14**, volume was reported as 0 with identical OHLC/close values, suggesting a possible market holiday or data anomaly.

Let me know if you'd like further analysis!


## Check

Explain what capability this step added and which earlier limitation it fixes.